In [1]:
import os
import sys
import time
from pathlib import Path

start = time.time()
print("[checkpoint] kernel 已启动")

# 兼容「从 notebooks 目录运行」与「从项目根目录运行」两种场景
project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 这里不再覆盖 BGE_LOCAL_FILES_ONLY，统一以 .env 配置为准
print(f"[checkpoint] project_root = {project_root}")
print(f"[checkpoint] BGE_LOCAL_FILES_ONLY={os.getenv('BGE_LOCAL_FILES_ONLY', '(unset)')}")

import json
import pandas as pd
print("[checkpoint] 基础依赖导入完成")

from src.retriever import DocumentRetriever
from src.pipelines import BasePipeline, RAGPipeline, RAGVerifyPipeline, RAGCovePipeline
print(f"[checkpoint] 项目模块导入完成，用时 {time.time() - start:.2f}s")

[checkpoint] kernel 已启动
[checkpoint] project_root = /Users/sunshine/Desktop/人工智能基础
[checkpoint] BGE_LOCAL_FILES_ONLY=0
[checkpoint] 基础依赖导入完成


/Users/sunshine/Desktop/人工智能基础/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


[checkpoint] 项目模块导入完成，用时 12.04s


In [2]:
# 1. 知识库入库
# 这一步可能比较耗时，尤其是第一次计算向量时。
print("[checkpoint] 开始初始化 retriever")
retriever = DocumentRetriever()
print("[checkpoint] 开始向量化入库")
retriever.ingest_document(str(project_root / "data" / "raw_docs" / "knowledge.txt"))

# 2. 实例化所有管道
print("[checkpoint] 开始实例化 pipelines")
pipelines = {
    "Base_LLM": BasePipeline(),
    "RAG": RAGPipeline(),
    "RAG_Verify": RAGVerifyPipeline(),
    "RAG_CoVe": RAGCovePipeline(),
}
print("[checkpoint] pipelines 实例化完成")

# 3. 读取测试集
with open(project_root / "data" / "test_dataset.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

results = []

# 4. 执行测试
print("实验开始运行...\n")
for item in test_data:
    question = item["question"]
    row = {"question": question, "ground_truth": item["ground_truth"]}

    for name, pipeline in pipelines.items():
        res = pipeline.run(question)
        row[f"{name}_answer"] = res["answer"]
        row[f"{name}_latency"] = res["latency"]

    results.append(row)

# 5. 展示结果表格
df = pd.DataFrame(results)
print("测试完成！详细结果如下：")
display(df)

[checkpoint] 开始初始化 retriever
[checkpoint] 开始向量化入库
向量化入库完成！
[checkpoint] 开始实例化 pipelines
[checkpoint] pipelines 实例化完成
实验开始运行...

测试完成！详细结果如下：


,question,ground_truth,Base_LLM_answer,Base_LLM_latency,RAG_answer,RAG_latency,RAG_Verify_answer,RAG_Verify_latency,RAG_CoVe_answer,RAG_CoVe_latency
0,DeepSeek V3 模型是在哪一年发布的？,2024年,DeepSeek V3 模型是在 **2024 年** 发布的。,1.367372,根据上下文，DeepSeek V3 于 2024 年底正式发布。,3.223038,根据上下文，DeepSeek V3 于 2024 年底正式发布。,3.019205,根据核查结果，原草稿中“DeepSeek V3 于 2024 年底正式发布”这一表述在上下文...,9.004039
1,该模型的训练成本是多少？,550万美元,该模型的训练成本尚未公开披露。,0.910868,根据上下文，该模型的训练总成本仅为550万美元。,2.477220,根据上下文，该模型的训练总成本为550万美元。,2.180445,根据核查结果，原草稿中的表述“该模型的训练总成本为550万美元”与上下文一致，且未发现其他来...,8.347672
2,根据规定，幻觉率高于多少算不达标？,20%,根据相关规定，幻觉率高于**5%** 通常被视为不达标。,0.974797,根据上下文，幻觉率如果高于 **20%** 则视为测试不达标。,4.513855,根据上下文，幻觉率如果高于 **20%** 则视为测试不达标。,2.334891,根据核查结果，原草稿中关于“幻觉率高于20%视为测试不达标”的表述缺乏外部来源或行业标准支持...,10.959521
3,DeepSeek V3 的首席架构师是谁？,上下文中未提及,DeepSeek V3 的首席架构师是 **周畅**（Chao Zhou）。,1.013415,抱歉，提供的信息不足以回答该问题。,3.470345,【拒答】抱歉，由于缺乏确凿证据，我拒绝回答该问题。,2.810676,根据核查结果，原草稿中的表述“提供的信息不足以回答该问题”缺乏可验证的事实或数据来源，且上下...,10.772733
